# Module 07-2: ModelRouter - 태스크별 모델 자동 라우팅 (솔루션)

## 학습 목표
- `ModelTier` Enum으로 모델 등급을 정의할 수 있다
- `RoutingRule` 데이터클래스로 라우팅 규칙을 명세할 수 있다
- `ModelRouter.get_model()` 메서드로 태스크와 입력 크기에 따라 모델을 선택할 수 있다
- `input_size` 기반 동적 업그레이드 로직을 구현할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/07-llm-call-optimization.md` 섹션 2.1

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import logging
from dataclasses import dataclass
from enum import Enum

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

print("환경 설정 완료")

## 실습 1: ModelTier Enum 정의 (솔루션)

In [ ]:
# 솔루션
class ModelTier(str, Enum):
    """모델 등급."""
    FAST = "fast"      # Haiku: 빠르고 저렴
    SMART = "smart"    # Sonnet: 정밀하고 고성능


print(f"ModelTier 정의: {[t.value for t in ModelTier]}")

In [ ]:
# 검증 셀
assert hasattr(ModelTier, 'FAST'), "ModelTier.FAST가 없습니다"
assert hasattr(ModelTier, 'SMART'), "ModelTier.SMART가 없습니다"
assert ModelTier.FAST != ModelTier.SMART, "FAST와 SMART는 달라야 합니다"
assert isinstance(ModelTier.FAST, str), "ModelTier는 str을 상속해야 합니다"
print(f"FAST: {ModelTier.FAST.value}, SMART: {ModelTier.SMART.value}")
print("실습 1 완료!")

## 실습 2: RoutingRule 데이터클래스 정의 (솔루션)

In [ ]:
# 솔루션
@dataclass
class RoutingRule:
    """라우팅 규칙."""
    task_name: str
    default_tier: ModelTier
    upgrade_threshold: int | None = None


print("RoutingRule 정의 완료")

In [ ]:
# 검증 셀
rule = RoutingRule(task_name="analyze", default_tier=ModelTier.FAST, upgrade_threshold=5120)
assert rule.task_name == "analyze", "task_name이 올바르지 않습니다"
assert rule.default_tier == ModelTier.FAST, "default_tier가 올바르지 않습니다"
assert rule.upgrade_threshold == 5120, "upgrade_threshold가 올바르지 않습니다"

rule_no_threshold = RoutingRule(task_name="classify", default_tier=ModelTier.FAST)
assert rule_no_threshold.upgrade_threshold is None, "upgrade_threshold 기본값이 None이어야 합니다"
print(f"Rule: {rule}")
print("실습 2 완료!")

## 실습 3: ModelRouter 클래스 구현 (솔루션)

In [ ]:
# 솔루션
class ModelRouter:
    """태스크 복잡도 기반 LLM 모델 라우터."""

    def __init__(
        self,
        fast_model: str = "claude-haiku-4-5-20250514",
        smart_model: str = "claude-sonnet-4-5-20250514",
        temperature: float = 0.1,
        rules: list[RoutingRule] | None = None,
    ):
        self._fast_model = fast_model
        self._smart_model = smart_model
        self._temperature = temperature

        default_rules = [
            RoutingRule("classify", ModelTier.FAST),
            RoutingRule("summarize", ModelTier.FAST),
            RoutingRule("analyze", ModelTier.FAST, upgrade_threshold=5120),
            RoutingRule("generate_code", ModelTier.SMART),
            RoutingRule("generate_test", ModelTier.SMART),
            RoutingRule("evaluate", ModelTier.SMART),
        ]
        rule_list = rules if rules is not None else default_rules
        self._rules: dict[str, RoutingRule] = {r.task_name: r for r in rule_list}
        self._cache: dict[str, dict] = {}

    def get_model(self, task_name: str, input_size: int = 0) -> dict:
        """태스크에 적합한 모델 정보를 반환한다.

        Args:
            task_name: 태스크 이름 (라우팅 테이블에서 조회)
            input_size: 입력 데이터 크기 (바이트). 동적 업그레이드 판단용.

        Returns:
            {"model": 모델이름, "tier": ModelTier} 딕셔너리
        """
        rule = self._rules.get(task_name)
        if rule is None:
            tier = ModelTier.SMART
        else:
            tier = rule.default_tier
            if (tier == ModelTier.FAST
                    and rule.upgrade_threshold is not None
                    and input_size > rule.upgrade_threshold):
                tier = ModelTier.SMART

        model_name = self._fast_model if tier == ModelTier.FAST else self._smart_model
        return {"model": model_name, "tier": tier}


print("ModelRouter 클래스 정의 완료")

In [ ]:
# 검증 셀: 기본 라우팅
router = ModelRouter()

# 간단한 분류 → FAST
r_classify = router.get_model("classify")
assert r_classify is not None, "get_model이 None을 반환합니다"
assert r_classify["tier"] == ModelTier.FAST, f"classify는 FAST여야 합니다. 현재: {r_classify['tier']}"
assert "haiku" in r_classify["model"].lower(), f"FAST 모델에 'haiku'가 포함되어야 합니다. 현재: {r_classify['model']}"

# 코드 생성 → SMART
r_code = router.get_model("generate_code")
assert r_code["tier"] == ModelTier.SMART, f"generate_code는 SMART여야 합니다. 현재: {r_code['tier']}"
assert "sonnet" in r_code["model"].lower(), f"SMART 모델에 'sonnet'이 포함되어야 합니다. 현재: {r_code['model']}"

print(f"classify → {r_classify['model']} ({r_classify['tier']})")
print(f"generate_code → {r_code['model']} ({r_code['tier']})")
print("기본 라우팅 검증 완료!")

In [ ]:
# 검증 셀: 동적 업그레이드
r_small = router.get_model("analyze", input_size=1000)   # 1KB → FAST
r_large = router.get_model("analyze", input_size=10000)  # 10KB → SMART

assert r_small["tier"] == ModelTier.FAST, f"작은 입력 분석은 FAST여야 합니다. 현재: {r_small['tier']}"
assert r_large["tier"] == ModelTier.SMART, f"큰 입력 분석은 SMART여야 합니다. 현재: {r_large['tier']}"

print(f"analyze (1KB) → {r_small['tier']} ({r_small['model']})")
print(f"analyze (10KB) → {r_large['tier']} ({r_large['model']})")
print("동적 업그레이드 검증 완료!")

In [ ]:
# 검증 셀: 알 수 없는 태스크는 SMART로 폴백
r_unknown = router.get_model("unknown_task_xyz")
assert r_unknown["tier"] == ModelTier.SMART, f"알 수 없는 태스크는 SMART여야 합니다. 현재: {r_unknown['tier']}"
print(f"unknown_task → {r_unknown['tier']} (SMART 폴백 확인)")
print("실습 3 완료!")

## 실습 4: 커스텀 라우팅 규칙 및 비용 분석 (솔루션)

In [ ]:
# 솔루션
custom_rules = [
    RoutingRule("code_review", ModelTier.FAST, upgrade_threshold=8192),
    RoutingRule("architecture_analysis", ModelTier.SMART),
    RoutingRule("quick_check", ModelTier.FAST),
]
custom_router = ModelRouter(rules=custom_rules)

print("커스텀 라우터 생성 완료")
print("등록된 규칙:", list(custom_router._rules.keys()))

In [ ]:
# 비용 절감 분석
PRICING = {
    "haiku": {"input": 1.00, "output": 5.00},
    "sonnet": {"input": 3.00, "output": 15.00},
}

def calc_cost(tier: ModelTier, input_tokens: int, output_tokens: int) -> float:
    price = PRICING["haiku"] if tier == ModelTier.FAST else PRICING["sonnet"]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

# 시나리오: 하루 1,000건 분류 + 500건 코드 생성 + 200건 코드 분석
scenarios = [
    {"task": "classify", "count": 1000, "input_tokens": 500, "output_tokens": 50},
    {"task": "generate_code", "count": 500, "input_tokens": 1000, "output_tokens": 500},
    {"task": "analyze", "count": 200, "input_tokens": 3000, "output_tokens": 300},
]

print("=== ModelRouter 비용 절감 분석 ===")
print(f"{'태스크':<20} {'전부 Sonnet':<15} {'ModelRouter':<15} {'절감율':<10}")
print("-" * 60)

total_sonnet = 0
total_router = 0

for s in scenarios:
    r = router.get_model(s["task"])
    cost_sonnet = calc_cost(ModelTier.SMART, s["input_tokens"], s["output_tokens"]) * s["count"]
    cost_router = calc_cost(r["tier"], s["input_tokens"], s["output_tokens"]) * s["count"]
    savings = (1 - cost_router / cost_sonnet) * 100 if cost_sonnet > 0 else 0
    
    total_sonnet += cost_sonnet
    total_router += cost_router
    print(f"{s['task']:<20} ${cost_sonnet:<14.2f} ${cost_router:<14.2f} {savings:.0f}%")

total_savings = (1 - total_router / total_sonnet) * 100
print("-" * 60)
print(f"{'합계':<20} ${total_sonnet:<14.2f} ${total_router:<14.2f} {total_savings:.0f}%")

In [ ]:
# 검증 셀
assert custom_router is not None, "custom_router가 생성되어야 합니다"
assert len(custom_router._rules) >= 2, "최소 2개의 커스텀 규칙이 필요합니다"
assert total_router < total_sonnet, "ModelRouter는 전부 Sonnet보다 비용이 적어야 합니다"
print(f"비용 절감 효과: {total_savings:.1f}%")
print("실습 4 완료!")

## 정리

1. **ModelTier Enum**: FAST(Haiku)와 SMART(Sonnet) 두 등급으로 단순화
2. **RoutingRule 데이터클래스**: 태스크별 기본 등급과 업그레이드 임계값 명세
3. **ModelRouter**: 태스크명과 입력 크기로 최적 모델을 자동 선택
4. **동적 업그레이드**: input_size 기반으로 필요할 때만 고성능 모델 사용
5. **비용 분석**: 라우팅으로 약 51% 비용 절감 가능